# 面试问题：Evaluator–Optimizer 反思循环怎样设计才不会自嗨？

可以直接复述的回答是：第一，Evaluator 必须使用可验证 rubric，而不是泛泛地判断“好不好”。第二，事实一致性和不承诺未知结果应作为硬门禁。第三，Optimizer 只能根据反馈修改草稿，不能创造新事实。第四，每轮保存草稿、分项得分、反馈和停止原因。第五，需要最大轮数和无改进停止条件。第六，用同一批工单比较首稿与修订稿的分项通过率。下面用支付客服回复演示一个不调用 LLM 的确定性循环。

## 真实案例：支付客服 Agent 修订回复草稿

五条脱敏支付工单包含状态、预计时间和允许的下一步动作。评价维度是事实依据、可执行下一步、同理语气和禁止无依据保证。案例用模板模拟生成器，重点展示循环控制与可观测性；真实生产仍需模型和人工标注数据。

In [1]:
tickets = [  # 定义五条具有权威事实和动作的支付客服工单
    {"id": "CS-601", "issue": "银行卡重复扣款", "status": "已确认重复交易", "eta": "3 个工作日", "action": "提交退款", "reference": "PAY-91"},  # 可依据交易号发起退款
    {"id": "CS-602", "issue": "转账一直处理中", "status": "银行通道处理中", "eta": "24 小时", "action": "超过时限后再次查询", "reference": "PAY-92"},  # 不能承诺立即到账
    {"id": "CS-603", "issue": "退款未到账", "status": "退款已提交银行", "eta": "1 至 5 个工作日", "action": "保留退款编号", "reference": "REF-33"},  # 需要说明银行处理窗口
    {"id": "CS-604", "issue": "支付失败但余额被占用", "status": "预授权待释放", "eta": "7 个工作日内", "action": "若超时联系发卡行", "reference": "PAY-94"},  # 需要给出超时后的明确动作
    {"id": "CS-605", "issue": "陌生交易申诉", "status": "等待持卡人确认", "eta": "尚未确定", "action": "冻结卡片并提交争议", "reference": "PAY-95"},  # 安全优先且不能虚构完成时间
]  # 结束五条脱敏客服输入
print("客服输入：id | issue | status | eta | action | reference")  # 展示回复生成器可使用的权威字段
for ticket in tickets:  # 逐条输出五个业务工单
    print(f"{ticket['id']} | {ticket['issue']} | {ticket['status']} | {ticket['eta']} | {ticket['action']} | {ticket['reference']}")  # 呈现事实和允许动作


客服输入：id | issue | status | eta | action | reference
CS-601 | 银行卡重复扣款 | 已确认重复交易 | 3 个工作日 | 提交退款 | PAY-91
CS-602 | 转账一直处理中 | 银行通道处理中 | 24 小时 | 超过时限后再次查询 | PAY-92
CS-603 | 退款未到账 | 退款已提交银行 | 1 至 5 个工作日 | 保留退款编号 | REF-33
CS-604 | 支付失败但余额被占用 | 预授权待释放 | 7 个工作日内 | 若超时联系发卡行 | PAY-94
CS-605 | 陌生交易申诉 | 等待持卡人确认 | 尚未确定 | 冻结卡片并提交争议 | PAY-95


## Baseline / 基线：一次生成且不做分项检查

首稿模板追求安抚用户，统一承诺“马上到账”，既没有引用状态，也没有给出工单允许的下一步。

In [2]:
def baseline_draft(ticket):  # 实现不读取权威状态的一次生成基线
    return f"您的{ticket['issue']}我们已经处理，保证马上到账，请放心。"  # 使用无依据保证制造真实客服风险
baseline_drafts = {ticket["id"]: baseline_draft(ticket) for ticket in tickets}  # 为五条工单生成首稿
print("一次生成基线：id | draft")  # 输出未经评价的全部首稿
for ticket in tickets:  # 按输入顺序展示基线回复
    print(f"{ticket['id']} | {baseline_drafts[ticket['id']]}")  # 让重复承诺和缺少行动项可见


一次生成基线：id | draft
CS-601 | 您的银行卡重复扣款我们已经处理，保证马上到账，请放心。
CS-602 | 您的转账一直处理中我们已经处理，保证马上到账，请放心。
CS-603 | 您的退款未到账我们已经处理，保证马上到账，请放心。
CS-604 | 您的支付失败但余额被占用我们已经处理，保证马上到账，请放心。
CS-605 | 您的陌生交易申诉我们已经处理，保证马上到账，请放心。


## 核心实现：可验证 Rubric、反馈与修订轨迹

Evaluator 检查四项：是否引用权威状态、是否包含允许动作、是否表达理解、是否出现“保证/马上到账/赔偿”等无依据承诺。Optimizer 只拼接输入中已有事实。

In [3]:
forbidden_promises = ("保证", "马上到账", "一定赔偿")  # 定义客服回复不可无依据使用的承诺词
def evaluate(ticket, draft):  # 按四项可验证 rubric 评价单条回复
    scores = {"grounded": int(ticket["status"] in draft and ticket["eta"] in draft), "actionable": int(ticket["action"] in draft), "empathetic": int("理解" in draft), "safe": int(not any(term in draft for term in forbidden_promises))}  # 逐项生成零一分数
    feedback = [name for name, passed in scores.items() if not passed]  # 只返回未通过的具体维度
    return scores, feedback  # 返回分项结果供优化器使用
def optimize(ticket, draft, feedback):  # 仅使用工单权威字段修订客服回复
    if feedback:  # 任一分项失败时重建有依据的回复
        return f"理解您遇到{ticket['issue']}的担心。当前状态：{ticket['status']}；预计时间：{ticket['eta']}。下一步：{ticket['action']}，参考号：{ticket['reference']}。"  # 组合可追溯状态、时间和动作
    return draft  # 全部分项通过时保持原草稿
focus = tickets[1]  # 选择转账处理中工单展示完整反思循环
current_draft = baseline_draft(focus)  # 以一次生成基线作为第零轮草稿
reflection_trace = []  # 保存每轮草稿、反馈和总分
for iteration in range(1, 4):  # 最多允许三轮以控制成本和死循环
    scores, feedback = evaluate(focus, current_draft)  # 评价当前草稿的四项 rubric
    reflection_trace.append({"iteration": iteration, "score": sum(scores.values()), "feedback": feedback, "draft": current_draft})  # 写入可重放反思轨迹
    if not feedback:  # 所有 rubric 通过时提前停止
        break  # 避免无意义继续改写
    current_draft = optimize(focus, current_draft, feedback)  # 根据明确失败项生成下一轮草稿
print("CS-602 反思轨迹：iteration | score/4 | feedback | draft")  # 输出每轮可观察的评价与修订
for step in reflection_trace:  # 逐轮展示停止前的完整状态
    print(f"{step['iteration']} | {step['score']}/4 | {step['feedback']} | {step['draft']}")  # 让得分提升原因可直接检查


CS-602 反思轨迹：iteration | score/4 | feedback | draft
1 | 0/4 | ['grounded', 'actionable', 'empathetic', 'safe'] | 您的转账一直处理中我们已经处理，保证马上到账，请放心。
2 | 4/4 | [] | 理解您遇到转账一直处理中的担心。当前状态：银行通道处理中；预计时间：24 小时。下一步：超过时限后再次查询，参考号：PAY-92。


## 失败案例与修正：关键词堆砌可以欺骗软评价器

一个草稿重复“理解、下一步、核实”，表面很像高质量回复，却承诺一定赔偿。只数关键词的软评价器会给高分；修正后事实与安全是独立硬门禁。

In [4]:
gaming_draft = "理解理解，我们会核实。下一步请等待，我们保证一定赔偿。"  # 构造会迎合表面关键词的危险回复
def soft_keyword_score(draft):  # 模拟只统计正向关键词的脆弱评价器
    return sum(term in draft for term in ("理解", "核实", "下一步"))  # 忽略事实和无依据承诺
soft_score = soft_keyword_score(gaming_draft)  # 计算可被关键词堆砌抬高的软分数
strict_scores, strict_feedback = evaluate(tickets[4], gaming_draft)  # 用事实与安全 rubric 重新评价危险草稿
strict_pass = all(strict_scores.values())  # 只有四项全部通过才允许发送
fixed_gaming_draft = optimize(tickets[4], gaming_draft, strict_feedback)  # 使用权威工单字段修订危险回复
print("投机草稿：", gaming_draft)  # 展示会欺骗软评价器的原文
print(f"软评价分数={soft_score}/3，严格评价={strict_scores}，允许发送={strict_pass}")  # 对照两种评价器的行为
print("修订草稿：", fixed_gaming_draft)  # 展示移除无依据保证后的回复


投机草稿： 理解理解，我们会核实。下一步请等待，我们保证一定赔偿。
软评价分数=3/3，严格评价={'grounded': 0, 'actionable': 0, 'empathetic': 1, 'safe': 0}，允许发送=False
修订草稿： 理解您遇到陌生交易申诉的担心。当前状态：等待持卡人确认；预计时间：尚未确定。下一步：冻结卡片并提交争议，参考号：PAY-95。


## 结果表：五条工单首稿与修订稿分项对照

In [5]:
baseline_scores = []  # 收集五条首稿的 rubric 总分
optimized_scores = []  # 收集五条修订稿的 rubric 总分
optimized_drafts = {}  # 保存最终可发送回复供回归测试
print("id | baseline_score | optimized_score | final_feedback")  # 输出逐工单改进表
for ticket in tickets:  # 在同一批五条工单上比较首稿和修订稿
    draft = baseline_drafts[ticket["id"]]  # 读取当前工单的一次生成首稿
    before_scores, before_feedback = evaluate(ticket, draft)  # 计算首稿的四项分数
    revised = optimize(ticket, draft, before_feedback)  # 根据失败维度修订回复
    after_scores, after_feedback = evaluate(ticket, revised)  # 重新评价修订稿
    baseline_scores.append(sum(before_scores.values()))  # 保存首稿总分
    optimized_scores.append(sum(after_scores.values()))  # 保存修订稿总分
    optimized_drafts[ticket["id"]] = revised  # 保存最终回复
    print(f"{ticket['id']} | {sum(before_scores.values())}/4 | {sum(after_scores.values())}/4 | {after_feedback}")  # 展示每条回复的改进和剩余问题
mean_baseline_score = sum(baseline_scores) / len(baseline_scores)  # 计算一次生成平均分
mean_optimized_score = sum(optimized_scores) / len(optimized_scores)  # 计算反思修订平均分
print(f"平均 rubric：baseline={mean_baseline_score:.2f}/4，optimized={mean_optimized_score:.2f}/4")  # 输出同一数据同一 rubric 下的汇总


id | baseline_score | optimized_score | final_feedback
CS-601 | 0/4 | 4/4 | []
CS-602 | 0/4 | 4/4 | []
CS-603 | 0/4 | 4/4 | []
CS-604 | 0/4 | 4/4 | []
CS-605 | 0/4 | 4/4 | []
平均 rubric：baseline=0.00/4，optimized=4.00/4


## 结果解读

CS-602 首轮只有 0 分，第二轮引用了“银行通道处理中、24 小时、超过时限后再次查询”，四项全部通过并停止。关键词投机草稿虽然软分为 3/3，严格评价仍因缺事实、缺允许动作和不安全承诺失败。反思循环的价值来自可验证 rubric 和证据约束，而不是让模型无限次“再想想”。

## 生产边界

生产系统需要人工标注的 rubric 校准集、模型版本对照、事实抽取器、敏感内容策略、最大成本预算和人工兜底。Evaluator 与 Optimizer 不应使用完全相同的提示和盲点；高风险支付结论必须引用权威交易状态。本例模板确定性很强，没有覆盖自然语言多样性、对抗提示和评价器漂移。

## 最小回归测试

In [6]:
assert len(tickets) >= 5  # 保证客服案例至少包含五条真实业务工单
assert len(reflection_trace) == 2  # 保证示例在修订通过后及时停止而不空转
assert reflection_trace[-1]["score"] == 4  # 保证最终草稿通过四项可验证 rubric
assert soft_score == 3 and strict_pass is False  # 保证投机草稿能暴露软评价器缺陷
assert mean_optimized_score > mean_baseline_score  # 保证同一评测集上的修订得分确实提升
assert all(not any(term in draft for term in forbidden_promises) for draft in optimized_drafts.values())  # 保证最终回复不含无依据承诺
